In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ealaxi/paysim1")

print("Path to dataset files:", path)

100%|██████████| 178M/178M [00:10<00:00, 18.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ealaxi/paysim1/versions/2


In [ ]:
import os

path = '/root/.cache/kagglehub/datasets/ealaxi/paysim1/versions/2'

print(os.path.exists(path))
print(os.listdir(path))

True
['PS_20174392719_1491204439457_log.csv']


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from xgboost import XGBClassifier
import joblib

PROJECT_DIR = ''

df = pd.read_csv('/kagglehub/datasets/ealaxi/paysim1/versions/2')


df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()


LEAKAGE_COLS = ['oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

df['type_encoded'] = (df['type'] == 'CASH_OUT').astype(int)

FEATURE_COLS = ['amount', 'type_encoded', 'step']
X = df[FEATURE_COLS].copy()
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

clf = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42,
)
clf.fit(X_train, y_train)

probs = clf.predict_proba(X_test)[:, 1]
preds = clf.predict(X_test)

print("ROC-AUC:", roc_auc_score(y_test, probs))
print(classification_report(y_test, preds))
print("\nFeature importances:")
for col, imp in sorted(zip(FEATURE_COLS, clf.feature_importances_), key=lambda x: -x[1]):
    print(f"  {col}: {imp:.4f}")

joblib.dump(clf, f'{PROJECT_DIR}/txn_level_classifier_xgb.pkl')
print(f"\nSaved to {PROJECT_DIR}/txn_level_classifier_xgb.pkl")

ROC-AUC: 0.9387059505721342
              precision    recall  f1-score   support

           0       1.00      0.91      0.95    552439
           1       0.03      0.83      0.05      1643

    accuracy                           0.91    554082
   macro avg       0.51      0.87      0.50    554082
weighted avg       1.00      0.91      0.95    554082


Feature importances:
  type_encoded: 0.4753
  step: 0.3346
  amount: 0.1901

Saved to /content/drive/MyDrive/fraud_detection_project/txn_level_classifier_xgb.pkl


In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve, classification_report

precisions, recalls, thresholds = precision_recall_curve(y_test, probs)

print(f"{'threshold':>10} {'precision':>10} {'recall':>10}")
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    preds_t = (probs >= t).astype(int)
    tp = ((preds_t == 1) & (y_test == 1)).sum()
    fp = ((preds_t == 1) & (y_test == 0)).sum()
    fn = ((preds_t == 0) & (y_test == 1)).sum()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f"{t:>10.2f} {prec:>10.4f} {rec:>10.4f}")

 threshold  precision     recall
      0.10     0.0062     0.9635
      0.20     0.0105     0.9270
      0.30     0.0148     0.9051
      0.40     0.0201     0.8740
      0.50     0.0275     0.8278
      0.60     0.0365     0.7845
      0.70     0.0518     0.7316
      0.80     0.0737     0.6701
      0.90     0.1249     0.5575
